In [ ]:

# 1. --- CONFIGURATION ---
# Base path for your local files
base_path = r"C:\Users\julio\OneDrive - FCT NOVA\Mestrado\5 ano\2º Semestre (Tese)\scripts\LC_prod\CleanProjectFiles"

files_to_process = [
    {"name": "ind_at_9.5Ghz_MEMO.csv", "freq": "9.5 GHz", "style": "-"},
    {"name": "ind_at_6Ghz_MEMOMEMO_PrimeSim_default_Measurements_history_1_20260602_17_18_30.19.csv", "freq": "6.0 GHz", "style": "--"}
]

# Plotting Styles
colors = {'TT': 'green', 'bcQ': 'blue', 'wcQ': 'red'}
labels_corner = {'TT': 'Typical', 'bcQ': 'Best Case', 'wcQ': 'Worst Case'}

# 2. --- HELPERS ---
def parse_unit_string(val):
    """Parses strings like '234.78p' into floats."""
    if pd.isna(val): return 0.0
    val_str = str(val).lower().strip()
    match = re.search(r"([0-9\.]+)", val_str)
    if match: return float(match.group(1))
    return 0.0

# 3. --- DATA PROCESSING & PLOTTING ---
fig, (ax_l, ax_q) = plt.subplots(2, 1, figsize=(12, 14))
fig.suptitle('Inductor Comparison: Inductance and Quality Factor', fontsize=18, fontweight='bold')

summary_data = []

for f_info in files_to_process:
    full_path = os.path.join(base_path, f_info["name"])
    freq_label = f_info["freq"]
    l_style = f_info["style"]
    
    if not os.path.exists(full_path):
        print(f"Skipping: {full_path} not found.")
        continue

    # Load and identify columns
    df = pd.read_csv(full_path)
    l_col = [c for c in df.columns if c.startswith('L_') and ':ac' in c][0]
    q_col = 'Q:ac'
    temp_col = [c for c in df.columns if 'temp' in c.lower()][0]

    # Process each corner
    for ik in ['_TT', '_bcQ', '_wcQ']:
        key = ik.replace('_', '')
        # Filter for the corner and sort by temperature
        sub = df[df['Corner'].str.contains(ik)].copy()
        sub = sub.sort_values(temp_col)
        
        if sub.empty: continue
        
        x = sub[temp_col].values
        y_l = sub[l_col].apply(parse_unit_string).values # pH
        y_q = sub[q_col].values
        
        full_label = f"{labels_corner[key]} @ {freq_label}"
        color = colors[key]

        # --- Plot L over T ---
        # Using k=1 for simulator-style linear transitions
        spline_l = make_interp_spline(x, y_l, k=1)
        x_smooth = np.linspace(x.min(), x.max(), 300)
        ax_l.plot(x_smooth, spline_l(x_smooth), color=color, linestyle=l_style, linewidth=2, label=full_label)
        ax_l.scatter(x, y_l, color=color, s=30, alpha=0.4)

        # --- Plot Q over T ---
        spline_q = make_interp_spline(x, y_q, k=1)
        ax_q.plot(x_smooth, spline_q(x_smooth), color=color, linestyle=l_style, linewidth=2, label=full_label)
        ax_q.scatter(x, y_q, color=color, s=30, alpha=0.4)
        
        # --- Capture 26°C Values ---
        # Find index closest to 26.0
        idx_26 = (sub[temp_col] - 26.0).abs().idxmin()
        summary_data.append({
            'Freq': freq_label,
            'Corner': key,
            'Actual_Temp': sub.loc[idx_26, temp_col],
            'L_pH': parse_unit_string(sub.loc[idx_26, l_col]),
            'Q': sub.loc[idx_26, q_col]
        })

# --- FORMATTING ---
ax_l.set_title("Inductance vs Temperature (L over T)", fontsize=14, fontweight='bold')
ax_l.set_ylabel("Inductance (pH)", fontsize=12)
ax_l.grid(True, linestyle='--', alpha=0.6)
ax_l.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
ax_l.legend(loc='upper right', fontsize=9, ncol=2)

ax_q.set_title("Quality Factor vs Temperature (Q over T)", fontsize=14, fontweight='bold')
ax_q.set_ylabel("Quality Factor (Q)", fontsize=12)
ax_q.set_xlabel("Temperature (°C)", fontsize=12)
ax_q.grid(True, linestyle='--', alpha=0.6)
ax_q.legend(loc='upper right', fontsize=9, ncol=2)

plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()

# --- 4. PRINT SUMMARY FOR 26°C ---
print(f"\n{'='*65}")
print(f"{'INDUCTOR PARAMETERS AT 26°C':^65}")
print(f"{'='*65}")
print(f"{'Frequency':<12} | {'Corner':<8} | {'Temp':<8} | {'L (pH)':<12} | {'Q factor':<10}")
print(f"{'-'*65}")

for row in summary_data:
    print(f"{row['Freq']:<12} | {row['Corner']:<8} | {row['Actual_Temp']:<8.1f} | {row['L_pH']:<12.2f} | {row['Q']:<10.2f}")
print(f"{'='*65}")